# Graph-based Gym Environments: MaxCut and VM placement

This notebook is organised as a small experimental walkthrough:

1. **MaxCut:** create weighted graphs, interact with `MaxCutEnv`, and inspect the rendered state.
2. **Virtual machine placement (VMP):** use the `VMPlacementEnv` with one fixed problem size and let the environment generate its graphs internally.
3. **Domain randomization:** vary scenario parameters across many environment instances.

The rollouts below use random valid actions as a simple smoke test, skipping optimization algorithms. The aim is to exercise the environments and collect comparable metrics.

## 0. Setup


In [ ]:
%matplotlib inline

import sys
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import networkx as nx

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd / "LaGCORL" if (cwd / "LaGCORL").is_dir() else cwd
if not (PROJECT_ROOT / "config.yaml").exists():
    raise FileNotFoundError("Set PROJECT_ROOT to the LaGCORL checkout.")
for entry in (str(PROJECT_ROOT.parent), str(PROJECT_ROOT), str(cwd)):
    if entry not in sys.path:
        sys.path.insert(0, entry)

from LaGCORL.envs.env_maxcut import MaxCutEnv
from LaGCORL.envs.env_vmp import VMPlacementEnv
from notebook_helpers_api import run_random_valid_vmp_episode, sample_vmp_config

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
print("Project root:", PROJECT_ROOT)


## 1. MaxCut

For MaxCut, a solution assigns every node to one of two partitions. An edge contributes its weight to the cut when its endpoints are assigned to different partitions. We first use a small weighted graph to make the setup easy to inspect, then reuse the larger graph from the original notebook to exercise the same environment on a larger instance.

In [ ]:
# Create a weighted graph in your notebook
G = nx.Graph()
G.add_weighted_edges_from([
    ("A", "B", 5),
    ("A", "C", 3),
    ("B", "C", 7),
    ("B", "D", 2),
    ("C", "D", 6),
])

# The environment infers the node count and maximum edge weight
env = MaxCutEnv(graph=G)

obs, info = env.reset()

### 1.1 Run a random valid-action rollout on the small graph

At each step, select a node that the environment considers available and move it to the opposite partition. The environment's `render()` method visualises the evolving assignment.

In [ ]:
obs, info = env.reset(seed=SEED)
terminated = truncated = False

while not (terminated or truncated):
    action = env.sample_valid_action()
    if action is None:
        break

    current_partition = env.graph.nodes[action]["x_dict"]["partition"]
    partition = 1 - current_partition

    obs, reward, terminated, truncated, info = env.step(action, partition)
    env.render(show=True)

plt.show()

### 1.2 Scale up the MaxCut instance

The next cell builds a larger weighted graph and creates a fresh `MaxCutEnv`.

In [ ]:
# Create a weighted graph in your notebook
G = nx.Graph()

G.add_weighted_edges_from([
    # Group 1
    ("A", "B", 5), ("A", "C", 3), ("A", "D", 7), ("A", "E", 4),
    ("B", "C", 7), ("B", "D", 2), ("B", "F", 6), ("B", "G", 3),
    ("C", "D", 6), ("C", "E", 8), ("C", "H", 4), ("C", "I", 5),
    ("D", "E", 3), ("D", "F", 9), ("D", "J", 4),
    ("E", "G", 6), ("E", "H", 2), ("E", "K", 7),
    ("F", "G", 5), ("F", "J", 8), ("F", "L", 3),
    ("G", "H", 7), ("G", "I", 4), ("G", "M", 6),
    ("H", "I", 9), ("H", "K", 5), ("H", "N", 3),
    ("I", "J", 6), ("I", "M", 8), ("I", "O", 4),
    ("J", "L", 7), ("J", "O", 5), ("J", "P", 3),
    ("K", "L", 4), ("K", "M", 9), ("K", "Q", 6),
    ("L", "M", 5), ("L", "P", 8), ("L", "R", 4),
    ("M", "N", 7), ("M", "O", 3), ("M", "S", 6),
    ("N", "O", 5), ("N", "Q", 8), ("N", "T", 4),
    ("O", "P", 9), ("O", "S", 5), ("O", "U", 7),
    ("P", "Q", 6), ("P", "R", 3), ("P", "V", 8),
    ("Q", "R", 7), ("Q", "S", 4), ("Q", "W", 5),
    ("R", "S", 9), ("R", "T", 6), ("R", "X", 3),
    ("S", "T", 5), ("S", "U", 8), ("S", "Y", 4),
    ("T", "U", 7), ("T", "V", 3), ("T", "Z", 6),

    # Additional connections across the graph
    ("A", "F", 6), ("A", "J", 4), ("B", "E", 5),
    ("C", "G", 8), ("D", "H", 4), ("E", "I", 7),
    ("F", "K", 5), ("G", "L", 9), ("H", "M", 6),
    ("I", "N", 3), ("J", "O", 8), ("K", "P", 4),
    ("L", "Q", 7), ("M", "R", 5), ("N", "S", 9),
    ("O", "T", 6), ("P", "U", 3), ("Q", "V", 8),
    ("R", "W", 4), ("S", "X", 7), ("T", "Y", 5),
    ("U", "Z", 9),

    # Longer-range connections
    ("A", "M", 8), ("B", "N", 4), ("C", "O", 6),
    ("D", "P", 5), ("E", "Q", 9), ("F", "R", 4),
    ("G", "S", 7), ("H", "T", 3), ("I", "U", 8),
    ("J", "V", 5), ("K", "W", 6), ("L", "X", 4),
    ("M", "Y", 7), ("N", "Z", 5),
])

# The environment infers the node count and maximum edge weight
env = MaxCutEnv(graph=G)

obs, info = env.reset()

### 1.3 Run the larger instance

This uses the same rollout logic as the small graph. 

In [ ]:
obs, info = env.reset(seed=SEED)
terminated = truncated = False
max_rendered_steps = 30
step_count = 0

while not (terminated or truncated):
    action = env.sample_valid_action()
    if action is None:
        break

    current_partition = env.graph.nodes[action]["x_dict"]["partition"]
    partition = 1 - current_partition
    obs, reward, terminated, truncated, info = env.step(action, partition)

    if step_count < max_rendered_steps:
        env.render(show=True)
    step_count += 1

plt.show()
print(f"Steps executed: {step_count}")

## 2. Virtual machine placement (VMP): one fixed problem size

In VMP, virtual machines (VMs) are assigned to physical machines (PMs) subject to resource-capacity constraints. The environment tracks multiple objectives, including utilisation, packing, power, load, latency, and security-related risk.

Unlike the MaxCut examples, we do not create or pass any NetworkX graph objects: `VMPlacementEnv` generates its allocation, traffic, and latency graphs internally from scalar configuration parameters.

In [ ]:
VMP_SIZE = {"n_pms": 8, "n_vms": 5, "n_tenants": 3}

vmp_env = VMPlacementEnv(
    **VMP_SIZE,
    revisiting=False,
)

obs, info = vmp_env.reset(seed=SEED)
print("Fixed VMP size:", VMP_SIZE)
print("Observation type:", type(obs))
print("Initial metrics:", vmp_env.current_metrics)

### 2.1 Roll out one VMP episode and inspect objective changes

The helper below samples uniformly from currently valid `(VM, PM)` placements. 

In [ ]:
vmp_result, vmp_history = run_random_valid_vmp_episode(vmp_env, seed=SEED)
display(vmp_history.tail())
print("Rollout summary:", vmp_result)


## 3. Domain randomisation for VMP

Domain randomisation creates multiple scenarios by varying environment parameters. Here we **keep the problem size fixed** (`8` PMs, `20` VMs, `3` tenants) and vary resource ranges, traffic/latency settings, power parameters, risk ranges, and objective coefficients.

Each `VMPlacementEnv` instance generates its own graphs internally. This means the loop varies scenario parameters without manually creating, passing, or storing NetworkX graph objects. 

### 3.1 Generate and evaluate many scenarios

The following loop instantiates several environments from sampled scalar parameters, runs a short random-valid-action episode in each, and records the sampled parameters alongside initial/final metrics. Failed initialisations are recorded instead of silently discarded, which makes it easier to tune parameter ranges.

In [ ]:
rng = np.random.default_rng(SEED)
N_SCENARIOS = 10
scenario_rows = []
failure_rows = []

scenario_id = 0

while scenario_id < N_SCENARIOS:
    config = sample_vmp_config(rng)

    try:
        scenario_env = VMPlacementEnv(**config, max_sweeps=100)
        result, history = run_random_valid_vmp_episode(
            scenario_env,
            seed=SEED + scenario_id,
            max_steps=VMP_SIZE["n_vms"],
        )

        row = {
            "scenario_id": scenario_id,
            **config,
            **result,
            "status": "ok",
        }
        scenario_rows.append(row)

        # Increment only after a successful scenario
        scenario_id += 1

    except Exception as exc:
        failure_rows.append({
            "scenario_id": scenario_id,
            "error_type": type(exc).__name__,
            "error": str(exc),
            **config,
        })
        # Retry the same scenario_id

domain_randomisation_df = pd.DataFrame(scenario_rows)
domain_randomisation_failures_df = pd.DataFrame(failure_rows)

print(f"Successful scenarios: {len(domain_randomisation_df)}/{N_SCENARIOS}")
print(f"Failed attempts: {len(domain_randomisation_failures_df)}")

display(domain_randomisation_df.head())

### 3.2 Compare outcomes across the randomised scenarios

The table makes it possible to inspect how scenario parameters relate to the final objective values. These are outcomes of random rollouts, not evidence that domain randomisation improves a learned policy; that requires training and evaluating the same policy across a held-out set of scenarios.

In [ ]:
metric_columns = [
    c for c in [
        "final_avg_util",
        "final_packing_efficiency",
        "final_avg_power",
        "final_avg_load",
        "final_avg_security_risk",
        "final_weighted_sum",
        "final_reward",
    ]
    if c in domain_randomisation_df.columns
]

if not domain_randomisation_df.empty and metric_columns:
    display(domain_randomisation_df[["scenario_id", *metric_columns]].describe().T)

    if "final_weighted_sum" in domain_randomisation_df:
        ax = domain_randomisation_df.plot(
            x="scenario_id", y="final_weighted_sum", marker="o",
            legend=False, title="Final weighted objective by randomised scenario"
        )
        ax.set_ylabel("Final weighted sum")
        plt.tight_layout()
        plt.show()